# Forest Plot: Time-Varying Cox PFS (Figure 2F)

PFS = first of radiology progression or death, censored at last follow-up. T=0 = first LOT start. Grade 1+ and Grade 3+ rows per toxicity, irAE modeled as a time-varying exposure (onset date splits each patient's follow-up into pre-/post-onset intervals).

The **figure itself is ICI cohort only**, but the underlying Cox models are run for **three cohort versions** (ICI, non-ICI, and all patients regardless of treatment) and all three are exported to CSV for reference, even though only ICI is plotted.

OS (supp) is out of scope for this notebook for now and is in the supplementary section

In [ ]:
%matplotlib inline
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450, "axes.unicode_minus": False,
})
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import CoxTimeVaryingFitter
warnings.filterwarnings("ignore")

TOXICITY_COLUMNS = ["adrenal_insufficiency", "colitis", "hyperthyroidism",
                    "hypothyroidism", "pneumonitis", "liver_toxicity"]
TOXICITY_DISPLAY = {
    "adrenal_insufficiency": "Adrenal Insufficiency", "colitis": "Colitis",
    "hyperthyroidism": "Hyperthyroidism", "hypothyroidism": "Hypothyroidism",
    "pneumonitis": "Pneumonitis", "liver_toxicity": "Liver Toxicity",
}
GRADE_TIER_LABEL = {"g1": "Grade 1+", "g3": "Grade 3+"}
MIN_EVENTS = 10

In [ ]:
# Utility functions
def standardize_mrn_series(mrn_series):
    def _clean(mrn):
        try:
            s = str(mrn).strip().strip("'" + '"').replace('P-','').replace('p-','').replace('MSK-','')
            digits = re.findall(r'\d+', s)
            return str(int(digits[0])).zfill(8) if digits else None
        except (ValueError, TypeError):
            return None
    return mrn_series.apply(_clean)

def normalize_progression_label(val):
    if pd.isna(val):
        return "Unknown"
    v = str(val).encode("ascii", errors="ignore").decode("ascii")
    v = ' '.join(v.strip().strip("'" + '"').lower().split()).replace('_',' ').replace('-',' ')
    if "intermediate" in v or "indeterminate" in v: return "Intermediate"
    if v in ("yes","y","true","1","positive") or v.startswith("yes"): return "Yes"
    if v in ("no","n","false","0","negative") or v.startswith("no"): return "No"
    return "Unknown"

def _bool_col(df, candidates):
    col = next((c for c in candidates if c in df.columns), None)
    if col is None: return pd.Series(0, index=df.index)
    return df[col].fillna(0).apply(lambda x: 1 if str(x).strip().upper() in ("1","TRUE","YES","Y") else 0)

In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'main'))
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load all source data
print("Loading progression data...")
prog_df = pd.read_csv(os.path.join(DATA_DIR, 'table_timeline_radiology_cancer_progression_predictions.csv'), low_memory=False)
prog_df["MRN"] = standardize_mrn_series(prog_df["MRN"])
prog_df = prog_df[prog_df["MRN"].notna()].copy()
prog_df["START_DATE"] = pd.to_datetime(prog_df["START_DATE"], errors="coerce")
prog_df = prog_df[prog_df["START_DATE"].notna()].copy()
prog_df["progression_label"] = prog_df["PROGRESSION"].apply(normalize_progression_label)
prog_df["progression_prob"] = pd.to_numeric(prog_df["PROGRESSION_PROBABILITY"], errors="coerce")
print(f"  {len(prog_df):,} records, {prog_df['MRN'].nunique():,} patients")

print("Loading LLM patient-level calls...")
llm_calls = pd.read_csv(os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv'), encoding="latin-1", low_memory=False)
llm_calls.columns = llm_calls.columns.str.strip()
llm_calls["mrn"] = standardize_mrn_series(llm_calls["mrn"])
llm_calls = llm_calls[llm_calls["mrn"].notna()].copy()
for t in TOXICITY_COLUMNS:
    if t in llm_calls.columns:
        llm_calls[t] = pd.to_numeric(llm_calls[t], errors="coerce").fillna(0).astype(int)
avail_tox = [t for t in TOXICITY_COLUMNS if t in llm_calls.columns]
llm_calls["any_irae"] = (llm_calls[avail_tox].sum(axis=1) > 0).astype(int)
valid_mrns = set(llm_calls["mrn"].dropna())
print(f"  {len(llm_calls):,} patients, {llm_calls['any_irae'].sum():,} with any irAE")

print("Loading LOT data...")
lot = pd.read_csv(os.path.join(DATA_DIR, 'regimen_lot(in).csv'), encoding="latin-1", low_memory=False)
lot.columns = lot.columns.str.strip()
lot["MRN"] = standardize_mrn_series(lot["MRN"])
lot = lot[lot["MRN"].notna() & lot["MRN"].isin(valid_mrns)].copy()
lot["APR_START_DTE"] = pd.to_datetime(lot["APR_START_DTE"], errors="coerce")
lot["APR_END_DTE"] = pd.to_datetime(lot["APR_END_DTE"], errors="coerce")
lot = lot[lot["APR_START_DTE"].notna()].sort_values(["MRN", "APR_START_DTE"])
lot["contains_ici"] = _bool_col(lot, ["CONTAINS_IMMUNO", "contains_immuno"])
lot["has_ctla4"] = _bool_col(lot, ["CONTAINS_CTLA4", "contains_ctla4", "CONTAINS_CTLA4_IMMUNO"])
lot["has_pdl1"] = _bool_col(lot, ["CONTAINS_NON_CTLA4_IMMUNO", "contains_non_ctla4_immuno"])
lot["has_chemo"] = _bool_col(lot, ["CONTAINS_CHEMO", "contains_chemo"])
lot["has_targeted"] = _bool_col(lot, ["CONTAINS_TARGETED", "contains_targeted"])
lot["has_hormone"] = _bool_col(lot, ["CONTAINS_HORMONE", "contains_hormone"])
lot["has_biologic"] = _bool_col(lot, ["CONTAINS_BIOLOGIC", "contains_biologic"])
baseline_df = lot.groupby("MRN").agg(
    first_lot_start=("APR_START_DTE", "first"), lot1_end=("APR_END_DTE", "first"),
    lot1_ici=("contains_ici", "first"), lot1_pdl1=("has_pdl1", "first"),
    lot1_ctla4=("has_ctla4", "first"), lot1_chemo=("has_chemo", "first"),
    lot1_targeted=("has_targeted", "first"), lot1_hormone=("has_hormone", "first"),
    lot1_biologic=("has_biologic", "first")
).reset_index()
for col in ["lot1_ici","lot1_pdl1","lot1_ctla4","lot1_chemo","lot1_targeted","lot1_hormone","lot1_biologic"]:
    baseline_df[col] = baseline_df[col].fillna(0).astype(int)
print(f"  LOT-1 ICI: {baseline_df['lot1_ici'].sum():,} | Non-ICI: {(baseline_df['lot1_ici']==0).sum():,}")

print("Loading patient data...")
pat = pd.read_csv(os.path.join(DATA_DIR, 'Patients.csv'), low_memory=False)
pat.columns = pat.columns.str.strip()
mrn_col = next((c for c in pat.columns if "MRN" in c.upper()), None)
death_col = next((c for c in pat.columns if "DEATH" in c.upper() or "DOD" in c.upper()), None)
lfu_col = next((c for c in pat.columns if any(k in c.upper() for k in ("LAST_CONTACT","LAST_FU","LAST_FOLLOW","LFU"))), None)
pat["MRN"] = standardize_mrn_series(pat[mrn_col])
pat = pat[pat["MRN"].notna() & pat["MRN"].isin(valid_mrns)].copy()
pat["death_date"] = pd.to_datetime(pat[death_col], errors="coerce") if death_col else pd.NaT
pat["last_followup_date"] = pd.to_datetime(pat[lfu_col], errors="coerce") if lfu_col else pd.NaT
pat_df = pat[["MRN", "death_date", "last_followup_date"]].drop_duplicates("MRN")
print(f"  {len(pat_df):,} patients | {pat_df['death_date'].notna().sum():,} deaths")

print("Loading covariates...")
COVARS_BACKBONE_PATH = os.path.join(TOX_TABLE_DIR, 'llm84k_pneumonitis_grade0_20260630.csv')
cov = pd.read_csv(COVARS_BACKBONE_PATH, low_memory=False)
cov["mrn"] = standardize_mrn_series(cov["mrn"])
cov = cov[cov["mrn"].notna() & cov["mrn"].isin(valid_mrns)].copy()
cov["lot"] = pd.to_numeric(cov["lot"], errors="coerce")
cov = cov.sort_values(["mrn", "lot"]).groupby("mrn").first().reset_index()
cov["age_at_dx"] = pd.to_numeric(cov["age_at_lot_start"], errors="coerce") if "age_at_lot_start" in cov.columns else np.nan
sex_col = next((c for c in cov.columns if c.lower() in ("gender","sex")), None)
if sex_col:
    cov["gender_male"] = cov[sex_col].fillna("").str.upper().str.startswith("M").astype(float)
    cov.loc[cov[sex_col].isna(), "gender_male"] = np.nan
else:
    cov["gender_male"] = np.nan
ct_col = next((c for c in cov.columns if "cancer_type" == c.lower()), None)
cov["cancer_type"] = cov[ct_col].fillna("Unknown") if ct_col else "Unknown"
bmi_col = next((c for c in cov.columns if "bmi" in c.lower()), None)
if bmi_col:
    cov["bmi"] = pd.to_numeric(cov[bmi_col], errors="coerce")
    cov.loc[(cov["bmi"] < 10) | (cov["bmi"] > 80), "bmi"] = np.nan
else:
    cov["bmi"] = np.nan
covar_df = cov[["mrn", "age_at_dx", "gender_male", "cancer_type", "bmi"]].copy()
print(f"  Covariates for {len(covar_df):,} patients")

In [ ]:
# Load grade data and onset dates
print("Loading grade data...")
grade_raw = None
gpath = os.path.join(DATA_DIR, 'grade_results_84k_FIXED_FP.csv')
if os.path.exists(gpath):
    grade_raw = pd.read_csv(gpath, encoding="latin-1", low_memory=False)
    grade_raw.columns = grade_raw.columns.str.strip()
    grade_raw.rename(columns={"adrenal insufficiency":"adrenal_insufficiency","liver toxicity":"liver_toxicity"}, inplace=True)
    grade_raw["mrn"] = standardize_mrn_series(grade_raw["mrn"])
    grade_raw["window_start"] = pd.to_datetime(grade_raw["window_start"], errors="coerce")
    grade_raw = grade_raw[grade_raw["mrn"].notna() & grade_raw["mrn"].isin(valid_mrns)].sort_values(["mrn","window_start"])
    print(f"  {len(grade_raw):,} grade records")

# Onset grades per patient
grade_onset = pd.DataFrame({"mrn": list(valid_mrns)})
if grade_raw is not None:
    for t in TOXICITY_COLUMNS:
        if t in grade_raw.columns:
            m = grade_raw[["mrn", t]].copy()
            m[t] = pd.to_numeric(m[t], errors="coerce").fillna(0).astype(int)
            m.loc[m[t] == 0, t] = np.nan
            fp = m.groupby("mrn")[t].first().fillna(0).astype(int).rename(f"{t}_onset_grade")
            grade_onset = grade_onset.merge(fp, left_on="mrn", right_index=True, how="left")
onset_grade_cols = [f"{t}_onset_grade" for t in TOXICITY_COLUMNS if f"{t}_onset_grade" in grade_onset.columns]
grade_onset[onset_grade_cols] = grade_onset[onset_grade_cols].fillna(0).astype(int)
grade_onset["max_onset_grade"] = grade_onset[onset_grade_cols].max(axis=1).astype(int) if onset_grade_cols else 0
print(f"  {(grade_onset['max_onset_grade'] > 0).sum():,} with grade > 0")

# G1+ onset dates
print("Loading window-level for G1+ onset dates...")
onset_df_g1 = pd.DataFrame(index=pd.Index(list(valid_mrns), name="mrn"))
wpath = os.path.join(DATA_DIR, 'llama_maverick_84k_patient_results.csv')
if os.path.exists(wpath):
    wdf = pd.read_csv(wpath, encoding="latin-1", low_memory=False)
    wdf.columns = wdf.columns.str.strip()
    wdf.rename(columns={"adrenal insufficiency":"adrenal_insufficiency","liver toxicity":"liver_toxicity"}, inplace=True)
    wdf["mrn"] = standardize_mrn_series(wdf["mrn"])
    wdf["window_start"] = pd.to_datetime(wdf["window_start"], errors="coerce")
    wdf = wdf[wdf["mrn"].notna() & wdf["mrn"].isin(valid_mrns) & wdf["window_start"].notna()].sort_values(["mrn","window_start"])
    for t in TOXICITY_COLUMNS:
        if t in wdf.columns:
            wdf[t] = pd.to_numeric(wdf[t], errors="coerce").fillna(0)
            pos = wdf[wdf[t] > 0.5][["mrn","window_start"]].drop_duplicates("mrn")
            onset_df_g1 = onset_df_g1.join(pos.set_index("mrn").rename(columns={"window_start": f"{t}_onset_date"}), how="left")
    od_cols = [f"{t}_onset_date" for t in TOXICITY_COLUMNS if f"{t}_onset_date" in onset_df_g1.columns]
    if od_cols: onset_df_g1["any_irae_onset_date"] = onset_df_g1[od_cols].min(axis=1)
    print(f"  G1+ onsets: {onset_df_g1['any_irae_onset_date'].notna().sum():,}")

# G3+ onset dates
onset_df_g3 = pd.DataFrame(index=pd.Index(list(valid_mrns), name="mrn"))
if grade_raw is not None:
    for t in TOXICITY_COLUMNS:
        if t in grade_raw.columns:
            g3 = grade_raw[pd.to_numeric(grade_raw[t], errors="coerce").fillna(0) >= 3]
            p3 = g3[["mrn","window_start"]].drop_duplicates("mrn")
            onset_df_g3 = onset_df_g3.join(p3.set_index("mrn").rename(columns={"window_start": f"{t}_onset_date"}), how="left")
    od3 = [f"{t}_onset_date" for t in TOXICITY_COLUMNS if f"{t}_onset_date" in onset_df_g3.columns]
    if od3: onset_df_g3["any_irae_onset_date"] = onset_df_g3[od3].min(axis=1)
    print(f"  G3+ onsets: {onset_df_g3['any_irae_onset_date'].notna().sum():,}")

In [ ]:
def build_cohort(ici_filter="ici"):
    """ici_filter: 'ici' (LOT-1 contains immunotherapy), 'non_ici' (does not),
    or 'all' (no filter on LOT-1 treatment composition)."""
    pos_prog = prog_df[prog_df["progression_label"].isin(["Yes", "Intermediate"])].copy()
    first_prog = pos_prog.sort_values("START_DATE").groupby("MRN").agg(
        progression_date=("START_DATE", "first"),
        n_progression_events=("START_DATE", "count")
    ).reset_index()

    coh = llm_calls[["mrn", "any_irae"] + avail_tox].merge(
        grade_onset[["mrn"] + onset_grade_cols + ["max_onset_grade"]], on="mrn", how="left")
    for gc in onset_grade_cols + ["max_onset_grade"]:
        coh[gc] = coh[gc].fillna(0).astype(int)
    coh = coh.merge(baseline_df.rename(columns={"MRN": "mrn"}), on="mrn", how="inner")

    if ici_filter == "ici":
        coh = coh[coh["lot1_ici"] == 1].copy()
    elif ici_filter == "non_ici":
        coh = coh[coh["lot1_ici"] == 0].copy()
    # "all" -> no filter

    coh = coh[coh["first_lot_start"].notna()].copy()
    coh.rename(columns={"first_lot_start": "t0_date", "lot1_end": "lot1_end_date"}, inplace=True)
    coh = coh.merge(pat_df.rename(columns={"MRN": "mrn"}), on="mrn", how="left")
    coh = coh.merge(covar_df, on="mrn", how="left")
    coh = coh.merge(first_prog.rename(columns={"MRN": "mrn"})[["mrn","progression_date","n_progression_events"]], on="mrn", how="left")

    # PFS computation
    coh.loc[(coh["progression_date"] - coh["t0_date"]).dt.days < 0, "progression_date"] = pd.NaT
    coh["days_to_prog"] = (coh["progression_date"] - coh["t0_date"]).dt.days
    coh["days_to_death"] = (coh["death_date"] - coh["t0_date"]).dt.days
    coh["days_to_lfu"] = (coh["last_followup_date"] - coh["t0_date"]).dt.days
    coh.loc[coh["days_to_death"] < 0, "death_date"] = pd.NaT
    coh["days_to_death"] = (coh["death_date"] - coh["t0_date"]).dt.days

    prog_d = coh["days_to_prog"].fillna(np.inf).values
    death_d = coh["days_to_death"].fillna(np.inf).values
    lfu_d = coh["days_to_lfu"].fillna(0).values
    first_event = np.minimum(prog_d, death_d)
    has_event = (first_event <= lfu_d) & np.isfinite(first_event)
    coh["pfs_days"] = np.where(has_event, first_event, np.maximum(lfu_d, 0))
    coh["pfs_event"] = has_event.astype(int)
    coh["pfs_months"] = coh["pfs_days"] / 30.44
    coh = coh[coh["pfs_days"] > 0].copy()

    # Cancer type: kept as a raw categorical column for Cox STRATIFICATION, not dummy-encoded
    # as a fixed effect. This gives each cancer type its own baseline hazard rather than
    # assuming one shared baseline shape shifted by a constant HR per type.
    coh["cancer_type"] = coh["cancer_type"].fillna("Unknown").astype(str)
    return coh

cohort_versions = {
    "ici": build_cohort("ici"),
    "non_ici": build_cohort("non_ici"),
    "all": build_cohort("all"),
}
for v, c in cohort_versions.items():
    print(f"{v}: {len(c):,} patients | AE+: {c['any_irae'].sum():,} | PFS events: {c['pfs_event'].sum():,}")

In [ ]:
# Time-varying Cox helpers
def build_tv_rows(cohort, onset_months_series):
    covar_cols = [c for c in cohort.columns if c in
        ("age_at_dx","gender_male","bmi","lot1_chemo","lot1_targeted",
         "lot1_hormone","lot1_biologic","lot1_pdl1","lot1_ctla4")]
    strata_cols = ["cancer_type"]  # kept separate from covar_cols -- stratified, not adjusted for
    c = cohort[["mrn", "pfs_months", "pfs_event"] + covar_cols + strata_cols].copy()
    c["onset_months"] = onset_months_series.reindex(c.index).values
    c.loc[c["onset_months"] <= 0, "onset_months"] = np.nan
    c["has_onset"] = c["onset_months"].notna() & (c["onset_months"] < c["pfs_months"])
    rows = []
    exposed = c[c["has_onset"]].copy()
    unexposed = c[~c["has_onset"]].copy()
    if len(exposed):
        pre = exposed.copy()
        pre["start"] = 0.0; pre["stop"] = pre["onset_months"]; pre["pfs_event"] = 0; pre["irae_exposed"] = 0
        rows.append(pre[["mrn","start","stop","pfs_event","irae_exposed"] + covar_cols + strata_cols])
        post = exposed.copy()
        post["start"] = post["onset_months"]; post["stop"] = post["pfs_months"]; post["irae_exposed"] = 1
        rows.append(post[["mrn","start","stop","pfs_event","irae_exposed"] + covar_cols + strata_cols])
    if len(unexposed):
        unexposed["start"] = 0.0; unexposed["stop"] = unexposed["pfs_months"]; unexposed["irae_exposed"] = 0
        rows.append(unexposed[["mrn","start","stop","pfs_event","irae_exposed"] + covar_cols + strata_cols])
    if not rows: return pd.DataFrame()
    tv = pd.concat(rows, ignore_index=True)
    return tv[tv["stop"] > tv["start"]].copy()

def get_cox_covariates(tv):
    # cancer_type deliberately excluded -- it's passed to CoxTimeVaryingFitter via strata=,
    # not included as a fixed-effect covariate here.
    base = ["age_at_dx","gender_male","bmi","lot1_chemo","lot1_targeted","lot1_hormone","lot1_biologic"]
    return [c for c in base if c in tv.columns and tv[c].nunique() >= 2 and pd.to_numeric(tv[c], errors="coerce").notna().sum() > 0]

def slug(s):
    return (s.lower().replace(" ", "_").replace("(", "").replace(")", "")
            .replace("+", "plus").replace("-", "_"))

In [ ]:
all_full_results = []   # full per-covariate rows, all 3 cohort versions -- for CSV export
ici_pfs_results = []     # irae_exposed-only summary, ICI version -- for the plot

for version_name, coh in cohort_versions.items():
    n_events_total = int(coh["pfs_event"].sum())
    for tox in TOXICITY_COLUMNS:
        if tox not in coh.columns: continue
        for onset_df, tier in [(onset_df_g1, "g1"), (onset_df_g3, "g3")]:
            onset_col = f"{tox}_onset_date"
            if onset_col not in onset_df.columns: continue
            onset_months = (coh["mrn"].map(onset_df[onset_col]) - coh["t0_date"]).dt.days / 30.44
            tv = build_tv_rows(coh, onset_months)
            model_id = f"{slug(tox)}_{tier}_{version_name}"
            row_label = f"{TOXICITY_DISPLAY[tox]} ({GRADE_TIER_LABEL[tier]})"

            if len(tv) == 0:
                all_full_results.append(pd.DataFrame([{
                    "model_id": model_id, "cohort_version": version_name, "toxicity": TOXICITY_DISPLAY[tox],
                    "grade_tier": GRADE_TIER_LABEL[tier], "covariate": None, "n_total": 0, "n_events": 0,
                    "model_status": "skipped: no time-varying rows"}]))
                continue

            cov_cols = get_cox_covariates(tv)
            keep = ["mrn","start","stop","pfs_event","irae_exposed"] + cov_cols + ["cancer_type"]
            tv = tv[[c for c in keep if c in tv.columns]].dropna()
            tv = tv[tv["stop"] > tv["start"]].copy()
            # Strata with fewer than 2 patients, or with no variation in irae_exposed within the
            # stratum, contribute ~nothing to the irae_exposed estimate but can still slow/destabilize
            # the fit -- drop them rather than silently including dead weight.
            strat_sizes = tv.groupby("cancer_type")["mrn"].nunique()
            valid_strata = strat_sizes[strat_sizes >= 2].index
            tv = tv[tv["cancer_type"].isin(valid_strata)].copy()

            n_total = tv["mrn"].nunique()
            n_events = int(tv["pfs_event"].sum())

            if n_events < MIN_EVENTS or tv["irae_exposed"].nunique() < 2:
                all_full_results.append(pd.DataFrame([{
                    "model_id": model_id, "cohort_version": version_name, "toxicity": TOXICITY_DISPLAY[tox],
                    "grade_tier": GRADE_TIER_LABEL[tier], "covariate": None, "n_total": n_total, "n_events": n_events,
                    "model_status": "skipped: insufficient events or exposure variance"}]))
                print(f"  [{version_name}] {row_label}: insufficient data")
                continue

            formula = "irae_exposed + " + " + ".join(cov_cols) if cov_cols else "irae_exposed"
            try:
                ctv = CoxTimeVaryingFitter(penalizer=0.1)
                ctv.fit(tv, id_col="mrn", start_col="start", stop_col="stop", event_col="pfs_event",
                        formula=formula, strata=["cancer_type"])

                summary = ctv.summary.reset_index().rename(columns={"index": "covariate"})
                summary["hr"] = np.exp(summary["coef"])
                ci_cols = ctv.confidence_intervals_.columns
                summary["hr_lower_95"] = np.exp(summary["coef lower 95%"])
                summary["hr_upper_95"] = np.exp(summary["coef upper 95%"])
                summary["model_id"] = model_id
                summary["cohort_version"] = version_name
                summary["toxicity"] = TOXICITY_DISPLAY[tox]
                summary["grade_tier"] = GRADE_TIER_LABEL[tier]
                summary["n_total"] = n_total
                summary["n_events"] = n_events
                summary["model_status"] = "fit"
                all_full_results.append(summary)

                if "irae_exposed" in ctv.params_.index:
                    hr = float(np.exp(ctv.params_["irae_exposed"]))
                    ci_lo = float(np.exp(ctv.confidence_intervals_.loc["irae_exposed", ci_cols[0]]))
                    ci_hi = float(np.exp(ctv.confidence_intervals_.loc["irae_exposed", ci_cols[1]]))
                    p = float(ctv.summary.loc["irae_exposed", "p"])
                    print(f"  [{version_name}] {row_label}: HR={hr:.2f} ({ci_lo:.2f}-{ci_hi:.2f}), p={p:.4f}")
                    if version_name == "ici":
                        ici_pfs_results.append({"ae": row_label, "hr": hr, "ci_lower": ci_lo, "ci_upper": ci_hi,
                            "p_value": p, "n": n_total, "n_exposed": tv[tv["irae_exposed"]==1]["mrn"].nunique(),
                            "n_events": n_events_total})
            except Exception as exc:
                all_full_results.append(pd.DataFrame([{
                    "model_id": model_id, "cohort_version": version_name, "toxicity": TOXICITY_DISPLAY[tox],
                    "grade_tier": GRADE_TIER_LABEL[tier], "covariate": None, "n_total": n_total, "n_events": n_events,
                    "model_status": f"failed: {exc}"}]))
                print(f"  [{version_name}] FAILED {row_label}: {exc}")

full_results_df = pd.concat(all_full_results, ignore_index=True)
print(f"\nTotal ICI results for plot: {len(ici_pfs_results)}")
print(f"Total full-model rows (all cohort versions): {len(full_results_df)}")

In [ ]:
cols_order = ["model_id", "cohort_version", "toxicity", "grade_tier", "covariate",
              "hr", "hr_lower_95", "hr_upper_95", "p", "n_total", "n_events", "model_status"]
full_results_df = full_results_df[[c for c in cols_order if c in full_results_df.columns]]
os.makedirs(RESULTS_DIR, exist_ok=True)
full_results_df.to_csv(os.path.join(RESULTS_DIR, 'Forest_Cox_PFS_2F_cox_models.csv'), index=False)
full_results_df

In [ ]:
# Render forest plot -- ICI cohort only
if not ici_pfs_results:
    print("No results to plot")
else:
    df_res = pd.DataFrame(ici_pfs_results)
    df_res = df_res[df_res["n_exposed"] >= 10].copy()
    df_res["_base"] = df_res["ae"].str.replace(r"\s*\(Grade.*", "", regex=True).str.strip()
    df_res["_tier"] = np.where(df_res["ae"].str.contains("Grade 3", na=False), "G3+", "G1+")
    _TOX_ORDER = TOXICITY_COLUMNS + ["any_ae"]
    order_index = {TOXICITY_DISPLAY.get(t, t): i for i, t in enumerate(_TOX_ORDER)}
    order_index["Any irAE"] = len(TOXICITY_COLUMNS)
    df_res["_ord"] = df_res["_base"].map(lambda b: order_index.get(b, 99))
    df_res["_tord"] = (df_res["_tier"] == "G3+").astype(int)
    df_res = df_res.sort_values(["_ord", "_tord"]).reset_index(drop=True)
    n = len(df_res)

    fig, ax = plt.subplots(figsize=(3.5, 2.56))
    y = np.arange(n - 1, -1, -1)
    for i, (_, row) in enumerate(df_res.iterrows()):
        hr = row["hr"]
        lo = max(row["ci_lower"], 0.1)
        hi = min(row["ci_upper"], 10.0)
        col = "#E63946" if row["p_value"] < 0.05 else "#888888"
        ax.plot([lo, hi], [y[i], y[i]], color=col, lw=2.2, solid_capstyle="round")
        ax.plot(max(min(hr, 10), 0.1), y[i], "o", color=col, ms=8,
                markeredgecolor="white", markeredgewidth=0.5, zorder=5)

    ax.axvline(1, color="#404040", ls="--", lw=1.0, zorder=0)
    labels = [f"{r['_base']} {r['_tier']}" for _, r in df_res.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=6)
    ax.set_xscale("log")
    ax.set_xlim(0.1, 10)
    ax.set_xticks([0.1, 0.25, 0.5, 1, 2, 4, 10])
    ax.set_xticklabels(["0.1", "0.25", "0.5", "1", "2", "4", "10"], fontsize=6)
    ax.set_xlabel("Hazard Ratio (log scale)", fontsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.set_ylim(-0.7, n - 0.3)
    fig.subplots_adjust(left=0.42, right=0.97, top=0.97, bottom=0.15)


In [ ]:
# Save
os.makedirs(RESULTS_DIR, exist_ok=True)
with PdfPages(os.path.join(RESULTS_DIR, 'Forest_Cox_PFS_2F.pdf')) as pdf:
    pdf.savefig(fig, dpi=450, bbox_inches="tight")
plt.close(fig)
print("Saved: ../results/main/Forest_Cox_PFS_2F.pdf")